# Auslan sign-chat backend on Colab

Starts the chat API (`research/signtest/sign_chat_backend/`, branch `recognition`) on a Colab GPU and gives it a public URL for the frontend.

- **sign → text**: Uni-Sign, Arm A from `openasl_pose_only_slt.pth` fine-tuned on Auslan-Daily (Communication BLEU-4 18.03)
- **text → sign**: SignSparK fine-tuned on Auslan-Daily Communication, round 2 (hands from retrieved keyframes, σ=1 smoothing)
- **dialogue**: Qwen2.5-1.5B-Instruct (~3 GB), or Claude / an OpenAI-compatible server / echo

**Runtime**: A100 (or L4 with High-RAM). **Run all**: sections 1-4 bring the server up (~15 min on a fresh runtime: ~13 GB copied from Drive, ~4 min of model loading), section 5 tries it. Section 6 holds optional checks; each stops the server, runs, and starts it again.

**Measured on an A100 (2026-09-25)**, with Qwen-7B and fp32 generators: text turn 4.3 s (dialogue 0.24 s + signing 4.1 s); recognition 1.34 s per clip of 2.3 s; GPU peak 40.1 GB. Every experiment and its numbers: `EXPERIMENTS.md`.

## 1. Drive, GPU, code, packages

In [ ]:
import os, sys, glob, json, shutil, subprocess, time
import torch
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip() or 'NO GPU - switch the runtime to a GPU')
print(f'python {sys.version.split()[0]} | torch {torch.__version__} | GPU {GPU_GB:.0f} GB | CPUs {os.cpu_count()}')
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive'
WORK = f'{DRIVE}/auslan_work'
LOCAL = '/content/signchat_local'
os.makedirs(LOCAL, exist_ok=True)

def sh(cmd, **kw):
    """Run a shell command and fail loudly (a failing `!pip` line does not stop the notebook)."""
    r = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True, **kw)
    if r.returncode:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError(f'failed: {cmd}')
    return r.stdout

# The backend code: this repository's `recognition` branch. For a private repo add a Colab secret GITHUB_TOKEN.
REPO_URL = 'https://github.com/randlyoyo/FIT5120-TE38-SignLanguage.git'
BRANCH = 'recognition'
APP = '/content/app'
CODE = f'{APP}/research/signtest'            # sign_chat_backend/, unisign/, auslan_smplx/
BACKEND = f'{CODE}/sign_chat_backend'
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
url = REPO_URL.replace('https://', f'https://{token}@') if token else REPO_URL
current = sh(['git', '-C', APP, 'rev-parse', '--abbrev-ref', 'HEAD']).strip() if os.path.isdir(f'{APP}/.git') else None
if current != BRANCH or not os.path.isdir(BACKEND):           # missing, or an old clone of another branch
    shutil.rmtree(APP, ignore_errors=True)
    sh(['git', 'clone', '-q', '--depth', '1', '-b', BRANCH, url, APP])
else:
    sh(['git', '-C', APP, 'pull', '-q'])
print('backend at', sh(['git', '-C', APP, 'log', '-1', '--format=%h %s']))

In [ ]:
# The two research repos, pinned to the commits the models were trained with
UNISIGN, UNISIGN_COMMIT = '/content/Uni-Sign', 'eed438bcb49e30405cd6ccdfcccca330c134e830'
SSK, SSK_COMMIT = '/content/SignSparK', '22a0b4ec292233c117be09a273fd8577bbbf7d8c'
for path, repo, commit in [(UNISIGN, 'https://github.com/ZechengLi19/Uni-Sign.git', UNISIGN_COMMIT),
                           (SSK, 'https://github.com/JianHe0628/SignSparK.git', SSK_COMMIT)]:
    if not os.path.isdir(f'{path}/.git'):
        sh(['git', 'clone', '-q', repo, path])
    sh(['git', '-C', path, 'checkout', '-q', commit])

t0 = time.time()
sh([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{BACKEND}/requirements.txt', 'wandb==0.21.3', 'sacrebleu'])
sh('apt-get -qq install -y ffmpeg > /dev/null')

# onnxruntime for the pose model, on the GPU. Two traps, both silent (the server just runs pose on the CPU):
# rtmlib pulls in the CPU `onnxruntime`, which shadows `onnxruntime-gpu`; and pip's latest onnxruntime-gpu
# (1.30) is a CUDA 13 build, while torch here ships CUDA 12 libraries. So: remove both, install the pinned
# CUDA 12 build from requirements.txt, and check that its CUDA provider library really loads.
ORT_PIN = next(l.split('#')[0].strip() for l in open(f'{BACKEND}/requirements.txt') if l.startswith('onnxruntime-gpu'))
sh([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'onnxruntime', 'onnxruntime-gpu'])
sh([sys.executable, '-m', 'pip', 'install', '-q', ORT_PIN])
ORT_CHECK = r'''
import torch, os, ctypes, onnxruntime as ort
ort.preload_dlls()
ctypes.CDLL(os.path.join(os.path.dirname(ort.__file__), 'capi', 'libonnxruntime_providers_cuda.so'))
print(ort.__version__, ort.get_available_providers())
'''
print('onnxruntime:', sh([sys.executable, '-c', ORT_CHECK]).strip(), '- CUDA provider loads')
print(f'packages ready ({time.time() - t0:.0f}s)')

from huggingface_hub import snapshot_download
MT5 = f'{UNISIGN}/pretrained_weight/mt5-base'
snapshot_download('google/mt5-base', revision='2eb15465c5dd7f72a8f7984306ad05ebc3dd1e1f', local_dir=MT5,
                  allow_patterns=['*.json', '*.model', 'pytorch_model.bin'])
print('repos and packages ready')

## 2. Weights and data from Drive

~13 GB copied to local disk with 8 parallel reads, largest first: Uni-Sign (2.2 GB), the three SignSparK generators (3.1 GB each, 0.83B-parameter UNets in fp32; nothing in them is redundant), the retrieval bank (1.2 GB) and SMPL-X. Files already copied in this runtime are skipped.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
UNISIGN_RUN = 'arm_a__openasl_pose_only_slt__official_stage3__single_a100__bf16'
SMPLX_SRC = (glob.glob(f'{DRIVE}/smplx_models/**/SMPLX_NEUTRAL_2020.npz', recursive=True) or [None])[0]
WEIGHTS_LOCAL, BANK_LOCAL = f'{LOCAL}/signspark_ft_smooth', f'{LOCAL}/bank/AuslanDaily_train.lmdb'
bank_drive = f'{WORK}/smplx_full/lmdb_smooth/train/AuslanDaily_train.lmdb'
jobs = {'unisign.pt': f'{WORK}/runs/{UNISIGN_RUN}/checkpoint.pt', 'SMPLX_NEUTRAL_2020.npz': SMPLX_SRC,
        **{f'signspark_ft_smooth/{s}.pt': f'{WORK}/signspark_ft_smooth/final/{s}.pt' for s in ('hand', 'body', 'face')},
        **{f'bank/AuslanDaily_train.lmdb/{f}': f'{bank_drive}/{f}' for f in (os.listdir(bank_drive) if os.path.isdir(bank_drive) else [])}}
missing = [k for k, v in jobs.items() if not v or not os.path.exists(v)]
if missing or not any(k.startswith('bank/') for k in jobs):
    raise SystemExit(f'missing on Drive: {missing or ["the retrieval bank LMDB"]}')

def copy(item):
    dst, src = item
    out = f'{LOCAL}/{dst}'
    if os.path.exists(out) and os.path.getsize(out) == os.path.getsize(src):
        return None
    os.makedirs(os.path.dirname(out), exist_ok=True)
    t = time.time()
    shutil.copyfile(src, out + '.part')
    os.replace(out + '.part', out)
    mb, dt = os.path.getsize(out) / 2**20, max(time.time() - t, 1e-3)
    return f'  {dst}: {mb:,.0f} MB in {dt:.0f}s ({mb / dt:.0f} MB/s)'

t0 = time.time()
with ThreadPoolExecutor(8) as ex:
    for msg in ex.map(copy, sorted(jobs.items(), key=lambda kv: -os.path.getsize(kv[1]))):
        if msg:
            print(msg, flush=True)
print(f'weights ready ({time.time() - t0:.0f}s): {sh(["du", "-sh", LOCAL]).split()[0]} in {LOCAL}')

## 3. Config and public URL

- **dialogue**: `hf` = Qwen2.5-1.5B-Instruct (~3 GB; the 7B took ~15 GB for one-line replies); or `anthropic` (Claude API, Colab secret `ANTHROPIC_API_KEY`), `openai` (an OpenAI-compatible server), `echo` (the avatar signs back what it was told).
- **Memory**: one text encoder shared by hand / body / face (-4.4 GB); `USE_BF16` stores the three generators in bf16 (-4.7 GB), checked in 3b; the text encoder stays on the GPU with 24 GB+; pose extraction on the GPU in batches of 64 with cuDNN HEURISTIC; hand / body / face sampled in parallel; the skeleton mp4 is drawn after the reply.
- **Public URL**: a Cloudflare quick tunnel (no account; anyone with the URL can use the API while the runtime is up). It changes every time this cell runs.

In [ ]:
DIALOGUE = 'hf'
DIALOGUE_MODEL = {'hf': 'Qwen/Qwen2.5-1.5B-Instruct', 'anthropic': 'claude-opus-5'}.get(DIALOGUE, '')
USE_BF16 = True            # generators stored in bf16; section 3b checks it and falls back to fp32 if it fails
PORT = 8000

if DIALOGUE == 'anthropic':
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')

# tunnel first, so the server knows its public URL (restarted if this cell is run again)
if 'tunnel' in globals() and isinstance(tunnel, subprocess.Popen) and tunnel.poll() is None:
    tunnel.terminate()
if not os.path.exists('/content/cloudflared'):
    sh(['wget', '-q', '-O', '/content/cloudflared',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'])
    os.chmod('/content/cloudflared', 0o755)
tunnel_log = open('/content/tunnel.log', 'w')
tunnel = subprocess.Popen(['/content/cloudflared', 'tunnel', '--no-autoupdate', '--url', f'http://localhost:{PORT}'],
                          stdout=tunnel_log, stderr=subprocess.STDOUT,
                          start_new_session=True)       # interrupting another cell no longer kills it
PUBLIC_URL = None
import re
for _ in range(60):
    time.sleep(1)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/tunnel.log').read())
    if m:
        PUBLIC_URL = m.group(0)
        break
print('public URL:', PUBLIC_URL)

import yaml
CFG = {
    'device': 'cuda', 'mock': False, 'media_dir': '/content/signchat_media', 'public_base_url': PUBLIC_URL or '',
    'sign2text': {'unisign_repo': UNISIGN, 'mt5_path': MT5, 'checkpoint': f'{LOCAL}/unisign.pt',
                  'code_dir': f'{CODE}/unisign', 'pose_device': 'cuda', 'pose_batch': 64},
    'text2sign': {'signspark_repo': SSK, 'code_dir': f'{CODE}/auslan_smplx', 'weights_dir': WEIGHTS_LOCAL,
                  'bank_lmdb': BANK_LOCAL, 'smplx_npz': f'{LOCAL}/SMPLX_NEUTRAL_2020.npz', 'encoder_on_gpu': GPU_GB >= 24,
                  'share_text_encoder': True, 'weights_dtype': 'bfloat16' if USE_BF16 else 'float32'},
    'dialogue': {'backend': DIALOGUE, 'model': DIALOGUE_MODEL},
}
with open('/content/signchat.yaml', 'w') as fh:
    yaml.safe_dump(CFG, fh, sort_keys=False)
print(open('/content/signchat.yaml').read())

## 3b. One-time checks, before the server starts (they need its GPU memory)

- **Parallel signing** (`CHECK_PARALLEL`): hand, body and face sampled at the same time must give the sequential result. Passed on 2026-09-25 (identical; once a 2.4e-3 difference in one hand); set it to `False` to skip.
- **bf16 generators** (`CHECK_BF16`, only when `USE_BF16`): `scripts/check_bf16.py` compares fp32 and bf16 on 256 test clips per stream (the training evaluation) and on everyday sentences (joint positions in mm), against the model's own seed-to-seed variation. If bf16 moves the signing more than a change of seed does, the config falls back to fp32 here and the server starts in fp32. ~10 min; results go to `EXPERIMENTS.md` D5.

In [ ]:
CHECK_PARALLEL = False     # passed 2026-09-25
CHECK_BF16 = True
CHECK_ENV = dict(os.environ, SIGNCHAT_CONFIG='/content/signchat.yaml', WANDB_MODE='disabled',
                 TOKENIZERS_PARALLELISM='false', TQDM_DISABLE='1')

def run_script(name, *args, check=True):
    """Run one of sign_chat_backend/scripts with the server's config; print its result or its error."""
    p = subprocess.run([sys.executable, f'{BACKEND}/scripts/{name}', *args], cwd=BACKEND, env=CHECK_ENV,
                       capture_output=True, text=True)
    print(p.stdout[-8000:])
    if p.returncode and (check or not p.stdout.strip()):
        print(p.stderr[-4000:])
        raise RuntimeError(f'{name} failed')
    return p.returncode

if CHECK_PARALLEL:
    run_script('check_parallel.py')

if USE_BF16 and CHECK_BF16:
    # the test split, copied locally (LMDB reads over the Drive mount are slow and lock-sensitive)
    TEST_LMDB = f'{LOCAL}/lmdb_smooth'
    src = f'{WORK}/smplx_full/lmdb_smooth/test/AuslanDaily_test.lmdb'
    if not os.path.exists(f'{TEST_LMDB}/test/AuslanDaily_test.lmdb/data.mdb'):
        shutil.copytree(src, f'{TEST_LMDB}/test/AuslanDaily_test.lmdb', dirs_exist_ok=True)
    rc = run_script('check_bf16.py', '--test-lmdb', TEST_LMDB, '--n', '256', '--out', '/content/check_bf16.json', check=False)
    if rc:                                             # bf16 changed the signing more than a seed change: use fp32
        USE_BF16 = False
        CFG['text2sign']['weights_dtype'] = 'float32'
        with open('/content/signchat.yaml', 'w') as fh:
            yaml.safe_dump(CFG, fh, sort_keys=False)
        print('\n>>> bf16 did not pass: the server will run the generators in fp32')
print('generators:', CFG['text2sign']['weights_dtype'])

## 4. Start the server

In [ ]:
import requests
env = dict(CHECK_ENV)

def stop_server():
    global server
    if 'server' in globals() and isinstance(server, subprocess.Popen) and server.poll() is None:
        server.terminate(); server.wait(60)

def start_server():
    """(Re)start the API in its own session, so interrupting a cell never kills it; wait for the models."""
    global server, health
    stop_server()
    server = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'signchat.server:app', '--host', '0.0.0.0', '--port', str(PORT)],
                              cwd=BACKEND, env=env, stdout=open('/content/server.log', 'w'), stderr=subprocess.STDOUT,
                              start_new_session=True)
    t0 = time.time()
    while True:
        time.sleep(5)
        if server.poll() is not None:
            print(open('/content/server.log').read()[-5000:])
            raise SystemExit('server exited')
        try:
            health = requests.get(f'http://localhost:{PORT}/api/health', timeout=2).json()
            break
        except Exception:
            print(f'loading models ... {time.time() - t0:.0f}s', end='\r')
    print(f'models loaded in {time.time() - t0:.0f}s')
    print(json.dumps(health, indent=2))
    if health.get('pose_providers') and 'CUDAExecutionProvider' not in health['pose_providers']:
        print('WARNING: pose extraction is running on the CPU (slow sign input); see the onnxruntime line of section 1')
    print(sh(['nvidia-smi', '--query-gpu=memory.used,memory.total', '--format=csv,noheader']))
    print('API:', PUBLIC_URL, '| docs:', f'{PUBLIC_URL}/docs')

start_server()

## 5. Try it

The skeleton video of a reply is drawn after the reply comes back (`video_status: rendering`), so `show_video` waits a moment for it.

In [ ]:
from IPython.display import Video, display

def show_video(url, width=480, timeout=30):
    if not url:
        return
    path = '/content/signchat_media/' + url.rsplit('/', 1)[1]
    for _ in range(timeout * 4):
        if os.path.exists(path):
            return display(Video(path, embed=True, width=width))
        time.sleep(0.25)
    print('video not ready after', timeout, 's:', path)

r = requests.post(f'http://localhost:{PORT}/api/chat/text', json={'text': 'Hello, how are you?'}).json()
print(json.dumps(r, indent=2)[:2000])
show_video(r.get('reply', {}).get('sign', {}).get('video_url'))

### Sign input without uploading

`files.upload()` does not work in the VS Code Colab extension, so this takes 5 random clips from the Auslan-Daily **Communication test split** on Drive (saved to `/content/test_clips`), compares the recognised text with the reference, then runs one full signed turn (video → text → reply → signing).

In [ ]:
import zipfile, random
rows = [json.loads(l) for l in open(f'{WORK}/manifest.jsonl')]
test = [r for r in rows if r['dataset'] == 'auslandaily' and r['split'] == 'test' and 'comm' in r['subset'].lower()]
random.seed(0)
picks = random.sample(test, 5)
print(f'{len(test)} Communication test clips, testing {len(picks)}')

_found = {}
def find(path):
    """The manifest holds the paths used at extraction time; find the same file on Drive by name."""
    if os.path.exists(path):
        return path
    name = os.path.basename(path)
    if name not in _found:
        hits = glob.glob(f'{DRIVE}/Auslan-Daily/**/{name}', recursive=True) or glob.glob(f'{DRIVE}/**/{name}', recursive=True)
        if not hits:
            raise FileNotFoundError(name)
        _found[name] = hits[0]
    return _found[name]

os.makedirs('/content/test_clips', exist_ok=True)
def fetch(r):
    out = f"/content/test_clips/{r['uid']}.mp4"
    if not os.path.exists(out):
        if '::' in r['video']:
            arch, member = r['video'].split('::', 1)
            with zipfile.ZipFile(find(arch)) as z, z.open(member) as src, open(out, 'wb') as dst:
                shutil.copyfileobj(src, dst)
        else:
            shutil.copyfile(find(r['video']), out)
    return out

for r in picks:
    path = fetch(r)
    with open(path, 'rb') as fh:
        res = requests.post(f'http://localhost:{PORT}/api/translate/sign-to-text',
                            files={'video': (os.path.basename(path), fh)}, data={'mirrored': 'false'}).json()
    print(f"\nreference : {r['text']}\nrecognised: {res.get('text', res)}\n  {res.get('timings')}")

# one full signed turn: video in -> recognised text -> reply -> signing
path = fetch(picks[0])
display(Video(path, embed=True, width=360))
with open(path, 'rb') as fh:
    r = requests.post(f'http://localhost:{PORT}/api/chat/sign', files={'video': (os.path.basename(path), fh)},
                      data={'mirrored': 'false', 'session_id': 'colab-test'}).json()
print(json.dumps(r, indent=2)[:1500])
show_video(r.get('reply', {}).get('sign', {}).get('video_url'))

### Speed and memory

Five text turns, then recognition on the test clips above (run the previous cell first).

In [ ]:
import threading
_peak, _stop = [0], threading.Event()
def _poll():                                   # GPU memory in use, sampled every 0.2 s during the test
    while not _stop.is_set():
        try:
            _peak[0] = max(_peak[0], int(sh(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits']).split()[0]))
        except Exception:
            pass
        time.sleep(0.2)
threading.Thread(target=_poll, daemon=True).start()
# memory and speed, for the deployment question: 5 text turns, then recognition on the test clips
ts = []
for q in ['Good morning.', 'What is your name?', 'I am hungry, let us eat.', 'See you tomorrow.', 'Thank you very much.']:
    ts.append(requests.post(f'http://localhost:{PORT}/api/chat/text', json={'text': q}).json()['timings'])
import statistics
for k in ts[0]:
    print(f'{k:<10} mean {statistics.fmean(t[k] for t in ts):.2f}s  max {max(t[k] for t in ts):.2f}s')

# recognition: the same test clips again (models are warm now)
if 'picks' in globals():
    rs = []
    for r in picks:
        path = fetch(r)
        with open(path, 'rb') as fh:
            res = requests.post(f'http://localhost:{PORT}/api/translate/sign-to-text',
                                files={'video': (os.path.basename(path), fh)}, data={'mirrored': 'false'}).json()
        rs.append({**res['timings'], 'seconds of video': res['frames'] / 25})
    print('\nrecognition, per clip:')
    for k in rs[0]:
        print(f'{k:<17} mean {statistics.fmean(t[k] for t in rs):.2f}s  max {max(t[k] for t in rs):.2f}s')
else:
    print('\n(run the test-clip cell first for recognition timings)')
print('GPU memory now:', sh(['nvidia-smi', '--query-gpu=memory.used,memory.total', '--format=csv,noheader']).strip())
print(sh('free -g | head -2'))
_stop.set()
print(f'GPU memory peak during the test: {_peak[0]:,} MiB')

## 6. Optional checks

Run when something in the pipeline changed. The first two need GPU memory, so they stop the server and start it again at the end (~2 min); the profiler runs next to it. All use the test clips from section 5.

In [ ]:
# Does pose extraction's cuDNN setting change Uni-Sign's translations? (EXHAUSTIVE vs HEURISTIC, 2026-09-25: 4/5 identical)
stop_server()
try:
    run_script('check_pose_setting.py', '--clips', '/content/test_clips', '--manifest', f'{WORK}/manifest.jsonl')
finally:
    start_server()

In [ ]:
# Where pose extraction spends its time, per part and per cuDNN setting, and how much each setting moves the keypoints
run_script('profile_pose.py', '--clips', '/content/test_clips', '--code', f'{CODE}/unisign')

In [ ]:
# Parallel vs sequential signing (the same check as section 3b), with the server stopped for the GPU memory
stop_server()
try:
    run_script('check_parallel.py')
finally:
    start_server()

## 7. Server log

In [ ]:
print(open('/content/server.log').read()[-4000:])